# 10 - The Integrated Cell Dataset

**Owner:** Mostafa Elsherif
**Inputs:** `master_monthly.csv` (notebook 07) and the outputs of notebooks 08 and 09
**Outputs:**

| File | Rows | Key | What it is |
|---|---|---|---|
| `master_cell.csv` | 560 | `cell_id` | **Everything we know about each 1 km cell, in one row.** This is the only input the scoring step reads. |
| `master_cell_dictionary.csv` | one row per column | `column` | What every column means, its unit, its source dataset and the notebook that made it |

## Why a second integrated table

`master_monthly.csv` is our integrated dataset in **time**: one row per cell per month, 60,480 rows. It is the right shape for asking "what happened in this field in May 2021?".

But the tool gives **one recommendation per field**, not one per month. The scoring step needs, for each cell, everything at once: its typical year, its long-term trend, its soil, and what each of the 8 crops would need there. Spread across 9 files, that invites mistakes: a wrong join, a forgotten file, two versions of the same column.

So we build `master_cell.csv`: **one row per cell, every source in it**. The two tables are two views of the same integrated data:

| Table | One row = | Use |
|---|---|---|
| `master_monthly.csv` | a cell in a month | time series, seasons, trends |
| `master_cell.csv` | a cell | scoring and the web tool |

## Where each group of columns comes from

| Group | Columns | Built in | Original source |
|---|---|---|---|
| Position | `row`, `col`, `lat`, `lon` | 00 | our grid |
| Land signals | seasons, gaps, stability, 40-year trend, farmland loss, data quality | 08 | Sentinel-2, Landsat |
| Typical year | NDVI and open-farmland share for each month | 08 | Sentinel-2 |
| Terrain and soil | elevation, slope, 7 soil qualities, main soil limit | 05, 06b | SRTM, GAEZ |
| Crops, per cell | suitability, class, attainable yield, sowing day, cycle, heat factor, irrigation need - for 8 crops | 06, 09 | GAEZ |
| Water and heat by month | rain, irrigation need, maximum temperature for each month | 08 | GPM, SMAP, NASA POWER |
| What is actually grown | each crop's share of the real 2020 harvest, and its real yield | 09 | GAEZ (actual 2020) |

The last two groups are the same for every cell: rain, heat and the 2020 harvest are measured for the area as a whole (notebook 03 explains why SMAP cannot be split into 1 km cells). We still put them in every row, so that each row is complete on its own.

## Naming rule

Month columns end in `_m01` to `_m12`. Crop columns end in the crop name (`_wheat`, `_maize`, ...). Every column is listed, with its unit, in `master_cell_dictionary.csv`.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

REPO = Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parent                      # works for everyone, not just one laptop
P = REPO / "data" / "processed"

N_CELLS = 560
CROPS = ["wheat", "maize", "rice", "cotton", "alfalfa", "sugarbeet", "potato", "soybean"]

master = pd.read_csv(P / "master_monthly.csv")
sig    = pd.read_csv(P / "signals_cell.csv")
scm    = pd.read_csv(P / "signals_cell_month.csv")
sam    = pd.read_csv(P / "signals_area_month.csv")
ccell  = pd.read_csv(P / "crops_cell.csv")
ctab   = pd.read_csv(P / "crops_table.csv")
cact   = pd.read_csv(P / "crops_actual_2020.csv")

for name, d, n in [("master_monthly", master, 60480), ("signals_cell", sig, N_CELLS),
                   ("signals_cell_month", scm, N_CELLS * 12), ("signals_area_month", sam, 12),
                   ("crops_cell", ccell, N_CELLS), ("crops_table", ctab, 8)]:
    assert len(d) == n, f"{name}: {len(d)} rows, expected {n}"
    print(f"loaded {name:20s} {d.shape}")
print(f"loaded {'crops_actual_2020':20s} {cact.shape}")

loaded master_monthly       (60480, 76)
loaded signals_cell         (560, 28)
loaded signals_cell_month   (6720, 8)
loaded signals_area_month   (12, 13)
loaded crops_cell           (560, 34)
loaded crops_table          (8, 15)
loaded crops_actual_2020    (13, 6)


## Step 1 - Take the fixed columns from the integrated table, and check them

Some columns in `master_monthly.csv` do not change with time: elevation, slope, soil, and GAEZ crop suitability. In that table each of them is repeated 108 times per cell, once per month.

**If the merge in notebook 07 was correct, every repeat must be identical.** If any cell had two different soil values in two different months, a join went wrong somewhere. We test every fixed column before trusting it.

In [2]:
FIXED = (["elev_mean_m", "elev_min_m", "elev_max_m"] +
         [c for c in master.columns if c.startswith("soil_")] +
         [c for c in master.columns if c.startswith(("gaez_slope_", "slope_srtm_"))] +
         [f"{v}_{c}" for c in CROPS for v in ["si", "class", "yield"]])
missing_cols = [c for c in FIXED if c not in master.columns]
assert not missing_cols, f"not in master_monthly: {missing_cols}"

g = master.groupby("cell_id")[FIXED]
n_values = g.nunique(dropna=False)
bad = (n_values > 1).sum()
print(f"fixed columns checked: {len(FIXED)}")
print(f"columns with more than one value inside any cell: {int((bad > 0).sum())}")
assert (bad == 0).all(), f"these columns change over time inside a cell: {bad[bad > 0].to_dict()}"

fixed = g.first().reset_index()
print(f"fixed columns taken from master_monthly: {fixed.shape}")

fixed columns checked: 43
columns with more than one value inside any cell: 0
fixed columns taken from master_monthly: (560, 44)


## Step 2 - The typical year, as 24 columns

From notebook 08: for every cell and every month, the average NDVI over 2017-2024 and the share of the cell's farmland that was open (NDVI below 0.35). Turned from long format (12 rows per cell) into wide format (12 columns per measure), so each cell is one row.

In [3]:
ty_ndvi = scm.pivot(index="cell_id", columns="month", values="ndvi_clim")
ty_open = scm.pivot(index="cell_id", columns="month", values="low_cover_clim") * 100
ty_ndvi.columns = [f"ndvi_m{m:02d}" for m in ty_ndvi.columns]
ty_open.columns = [f"open_pct_m{m:02d}" for m in ty_open.columns]
typical = ty_ndvi.join(ty_open).reset_index()
assert typical.shape == (N_CELLS, 25)
print(f"typical year: {typical.shape}")
print(typical[["ndvi_m01", "ndvi_m05", "ndvi_m07", "ndvi_m10", "open_pct_m05", "open_pct_m10"]].median().round(3).to_string())

typical year: (560, 25)
ndvi_m01         0.864
ndvi_m05         0.445
ndvi_m07         0.792
ndvi_m10         0.583
open_pct_m05    45.729
open_pct_m10    21.521


## Step 3 - Water and heat by month, and what is actually grown

These are the same for every cell, so we build them once and attach them to every row.

In [4]:
area = {}
for _, r in sam.iterrows():
    m = int(r.month)
    area[f"rain_mm_m{m:02d}"]          = r.rain_mm
    area[f"irrig_need_m{m:02d}"]       = r.wilting_frac_rainfed
    area[f"t_max_c_m{m:02d}"]          = r.t_max_c
area["rain_mm_year"] = sam.rain_mm.sum()

cand = cact[cact.is_candidate].set_index("crop")
for c in CROPS:
    area[f"actual_share_pct_{c}"]  = cand.loc[c, "share_of_harvest_pct"]
    area[f"actual_yield_t_ha_{c}"] = cand.loc[c, "yield_t_ha"]
    area[f"season_{c}"]            = ctab.set_index("crop").loc[c, "season"]

area_row = pd.DataFrame([area])
print(f"area-level columns: {area_row.shape[1]}")
print(f"annual rain: {area['rain_mm_year']:.1f} mm")

area-level columns: 61
annual rain: 109.9 mm


## Step 4 - Assemble

Everything is joined on `cell_id`, starting from the 560 cells of the land signals table. As in notebook 07, every join must keep exactly 560 rows.

In [5]:
crop_cols = [c for c in ccell.columns if c != "cell_id"]
parts = [
    ("crops per cell (09)", ccell),
    ("fixed columns (07)",  fixed),
    ("typical year (08)",   typical),
]

mc = sig.copy()
for label, d in parts:
    clash = (set(mc.columns) & set(d.columns)) - {"cell_id"}
    assert not clash, f"{label}: column clash {clash}"
    before = len(mc)
    mc = mc.merge(d, on="cell_id", how="left")
    assert len(mc) == before == N_CELLS, f"{label}: rows changed"
    print(f"OK  joined {label:22s} -> {mc.shape}")

mc = pd.concat([mc, pd.DataFrame([area] * len(mc), index=mc.index)], axis=1)
print(f"OK  attached {len(area)} area-level columns -> {mc.shape}")

mc = mc.sort_values("cell_id").reset_index(drop=True)

OK  joined crops per cell (09)    -> (560, 61)
OK  joined fixed columns (07)     -> (560, 104)
OK  joined typical year (08)      -> (560, 128)
OK  attached 61 area-level columns -> (560, 189)


## Step 5 - Checks

Beyond the shape, we check that the pieces agree with each other where they overlap.

In [6]:
grid = pd.read_csv(P / "grid_1km.csv").set_index("cell_id").loc[mc.cell_id]

checks = {
    "560 rows":                                        len(mc) == N_CELLS,
    "cell_id unique and 1-560":                        mc.cell_id.is_unique and mc.cell_id.min() == 1 and mc.cell_id.max() == N_CELLS,
    "no column completely empty":                      not mc.isna().all().any(),
    "lat/lon match the grid":                          np.allclose(mc.lat.values, grid.lat.values) and np.allclose(mc.lon.values, grid.lon.values),
    "typical-year winter mean = winter_ndvi (08)":     np.allclose(mc[["ndvi_m12", "ndvi_m01", "ndvi_m02", "ndvi_m03"]].mean(axis=1), mc.winter_ndvi),
    "spring gap NDVI = lowest of Mar-Jun months":      np.allclose(mc[["ndvi_m03", "ndvi_m04", "ndvi_m05", "ndvi_m06"]].min(axis=1), mc.spring_gap_ndvi),
    "every crop has 7 per-cell columns":               all(sum(c.endswith("_" + k) and not c.startswith(("actual_", "season_")) for c in mc.columns) == 7 for k in CROPS),
}
for k, v in checks.items():
    print(f"  {'PASS' if v else 'FAIL'}  {k}")
assert all(checks.values()), "a check failed - do not use master_cell.csv"

miss = mc.isna().sum()
print(f"\ncolumns with any missing value: {(miss > 0).sum()} of {mc.shape[1]}")
print(miss[miss > 0].sort_values(ascending=False).to_string())

  PASS  560 rows
  PASS  cell_id unique and 1-560
  PASS  no column completely empty
  PASS  lat/lon match the grid
  PASS  typical-year winter mean = winter_ndvi (08)
  PASS  spring gap NDVI = lowest of Mar-Jun months
  PASS  every crop has 7 per-cell columns

columns with any missing value: 12 of 189
si_rice             40
si_alfalfa          40
si_wheat            40
si_maize            40
si_potato           40
si_soybean          40
irrig_mm_maize      22
irrig_mm_wheat      22
irrig_mm_alfalfa    22
irrig_mm_rice       22
irrig_mm_potato     22
irrig_mm_soybean    22


The missing values are the ones we already know and explained: the 40 cells where GAEZ has no suitability for six crops (notebook 06), and the 22 cells on the eastern edge where GAEZ irrigation need is zero and was set to empty (notebook 09). Nothing new is missing.

## Step 6 - The column dictionary

A table of 190 columns is only useful if every column can be looked up. We generate the dictionary from the column names, so it cannot drift away from the data.

In [7]:
def describe(col):
    crop = next((c for c in CROPS if col.endswith("_" + c)), None)
    mo = col[-2:] if "_m" in col and col[-3:-2] == "m" and col[-2:].isdigit() else None
    base = col[: -len(crop) - 1] if crop else (col[:-4] if mo else col)
    M = f" in month {int(mo)}" if mo else ""
    K = f" for {crop}" if crop else ""
    table = {
        "cell_id": ("", "cell number, 1-560", "grid", "00"),
        "row": ("", "grid row, 1 = north", "grid", "00"), "col": ("", "grid column, 1 = west", "grid", "00"),
        "lat": ("degrees", "latitude of the cell centre", "grid", "00"), "lon": ("degrees", "longitude of the cell centre", "grid", "00"),
        "farm_pct": ("0-1", "share of the cell that is farmland, 2017-2024", "Sentinel-2", "08"),
        "is_farm_cell": ("yes/no", "farm_pct is at least 0.5", "Sentinel-2", "08"),
        "winter_ndvi": ("NDVI", "mean NDVI December-March", "Sentinel-2", "08"),
        "summer_ndvi": ("NDVI", "mean NDVI June-August", "Sentinel-2", "08"),
        "two_seasons": ("yes/no", "both seasons at least 0.5 NDVI", "Sentinel-2", "08"),
        "spring_gap_month": ("month", "month with the lowest NDVI, March-June", "Sentinel-2", "08"),
        "spring_gap_ndvi": ("NDVI", "NDVI in the spring gap month", "Sentinel-2", "08"),
        "spring_gap_open_pct": ("%", "farmland open in the spring gap month", "Sentinel-2", "08"),
        "autumn_gap_month": ("month", "month with the lowest NDVI, August-November", "Sentinel-2", "08"),
        "autumn_gap_ndvi": ("NDVI", "NDVI in the autumn gap month", "Sentinel-2", "08"),
        "autumn_gap_open_pct": ("%", "farmland open in the autumn gap month", "Sentinel-2", "08"),
        "winter_cv": ("ratio", "year-to-year variation of winter NDVI, 2017-2024", "Sentinel-2", "08"),
        "summer_cv": ("ratio", "year-to-year variation of summer NDVI, 2017-2024", "Sentinel-2", "08"),
        "ls_trend_per_decade": ("NDVI/decade", "Theil-Sen trend of Feb-Mar NDVI, 1985-2024", "Landsat", "08"),
        "ls_trend_p": ("p-value", "Mann-Kendall test of that trend", "Landsat", "08"),
        "ls_trend_q": ("q-value", "trend p-value after false discovery rate correction", "Landsat", "08"),
        "ls_trend_class": ("label", "improving / stable / declining (q < 0.05)", "Landsat", "08"),
        "ls_n_years": ("years", "years with a Landsat value", "Landsat", "08"),
        "ls_farm_1985_94": ("0-1", "farmland share 1985-1994", "Landsat", "08"),
        "ls_farm_2015_24": ("0-1", "farmland share 2015-2024", "Landsat", "08"),
        "ls_farm_pct_change": ("0-1", "farmland share change; negative = village growth", "Landsat", "08"),
        "valid_pct_mean": ("0-1", "average cloud-free share per month, 2017-2024", "Sentinel-2", "08"),
        "ndvi_missing_months": ("months", "months 2017-2024 with no clear view", "Sentinel-2", "08"),
        "elev_mean_m": ("m", "mean elevation", "SRTM", "05"), "elev_min_m": ("m", "lowest elevation", "SRTM", "05"),
        "elev_max_m": ("m", "highest elevation", "SRTM", "05"),
        "soil_overall": ("0-1", "worst soil quality rating (1 = no limit)", "GAEZ v5", "06b"),
        "soil_nutrients": ("0-1", "nutrient availability rating", "GAEZ v5", "06b"),
        "soil_nutrient_retention": ("0-1", "nutrient retention rating", "GAEZ v5", "06b"),
        "soil_rooting": ("0-1", "rooting conditions rating", "GAEZ v5", "06b"),
        "soil_oxygen": ("0-1", "oxygen / drainage rating", "GAEZ v5", "06b"),
        "soil_salinity": ("0-1", "salinity and sodicity rating", "GAEZ v5", "06b"),
        "soil_lime_gypsum": ("0-1", "lime and gypsum rating", "GAEZ v5", "06b"),
        "soil_workability": ("0-1", "workability rating", "GAEZ v5", "06b"),
        "soil_water_share": ("0-1", "share of soil-map pixels that are water", "GAEZ v5", "06b"),
        "soil_limit_class": ("class", "most limiting soil quality, GAEZ class 1-11", "GAEZ v5", "06b"),
        "soil_limit_name": ("label", "most limiting soil quality, in words", "GAEZ v5", "06b"),
        "gaez_slope_flat_pct": ("%", "land with 0-0.5% slope", "GAEZ v5", "06b"),
        "gaez_slope_lt2_pct": ("%", "land under 2% slope", "GAEZ v5", "06b"),
        "gaez_slope_mean_pct": ("%", "mean slope from slope classes", "GAEZ v5", "06b"),
        "slope_srtm_mean_pct": ("%", "mean slope of 30 m pixels (at the data's noise level, see 06b)", "SRTM", "06b"),
        "slope_srtm_max_pct": ("%", "steepest 30 m pixel", "SRTM", "06b"),
        "mci_class": ("class", "GAEZ multi-cropping class with irrigation (8 = triple rice)", "GAEZ v5", "09"),
        "rain_mm_year": ("mm", "mean annual rain, 2016-2024", "GPM", "08"),
        # per crop
        "si": ("0-100", "GAEZ suitability index, irrigated high input", "GAEZ v5", "06"),
        "class": ("class", "GAEZ suitability class, current cropland (1 = best)", "GAEZ v5", "06"),
        "yield": ("kg DW/ha", "GAEZ attainable yield, current cropland", "GAEZ v5", "06"),
        "plant_doy": ("day of year", "GAEZ best sowing day (single crop, see 09)", "GAEZ v5", "09"),
        "cycle_days": ("days", "GAEZ length of growth cycle", "GAEZ v5", "09"),
        "thermal": ("0-10000", "GAEZ thermal yield factor (10000 = no loss)", "GAEZ v5", "09"),
        "irrig_mm": ("mm", "GAEZ net irrigation need over the cycle", "GAEZ v5", "09"),
        "actual_share_pct": ("%", "share of all irrigated harvested area, 2020", "GAEZ v5 actual", "09"),
        "actual_yield_t_ha": ("t/ha", "real yield 2020, harvested product", "GAEZ v5 actual", "09"),
        "season": ("label", "winter / summer / all-year, from the GAEZ cycle", "GAEZ v5", "09"),
        # per month
        "ndvi": ("NDVI", "mean NDVI, 2017-2024", "Sentinel-2", "08"),
        "open_pct": ("%", "farmland with NDVI below 0.35, 2017-2024", "Sentinel-2", "08"),
        "rain_mm": ("mm", "mean rain, 2016-2024", "GPM", "08"),
        "irrig_need": ("0-1", "SMAP wilting fraction without irrigation (irrigation need)", "SMAP", "08"),
        "t_max_c": ("C", "mean daily maximum temperature, 2016-2024", "NASA POWER", "08"),
    }
    unit, desc, src, nbk = table.get(base, table.get(col, ("?", "?", "?", "?")))
    group = ("crop" if crop else "month" if mo else "cell")
    return dict(column=col, group=group, unit=unit, description=desc + K + M, source=src, notebook=nbk)

dic = pd.DataFrame([describe(c) for c in mc.columns])
undocumented = dic[dic.description.str.startswith("?")].column.tolist()
print(f"columns documented: {len(dic) - len(undocumented)} of {len(dic)}")
assert not undocumented, f"undocumented columns: {undocumented}"
print("\ncolumns by source:")
print(dic.source.value_counts().to_string())
print("\nexamples:")
print(dic.sample(6, random_state=1)[["column", "unit", "source", "description"]].to_string(index=False))

columns documented: 189 of 189

columns by source:
source
GAEZ v5           79
Sentinel-2        39
GAEZ v5 actual    16
GPM               13
SMAP              12
NASA POWER        12
Landsat            8
grid               5
SRTM               5

examples:
                 column  unit     source                                                           description
         irrig_mm_wheat    mm    GAEZ v5                     GAEZ net irrigation need over the cycle for wheat
soil_nutrient_retention   0-1    GAEZ v5                                             nutrient retention rating
               ndvi_m09  NDVI Sentinel-2                                       mean NDVI, 2017-2024 in month 9
           open_pct_m05     % Sentinel-2                   farmland with NDVI below 0.35, 2017-2024 in month 5
            class_wheat class    GAEZ v5         GAEZ suitability class, current cropland (1 = best) for wheat
         irrig_need_m09   0-1       SMAP SMAP wilting fraction without irrig

## Step 7 - Save

In [8]:
mc.to_csv(P / "master_cell.csv", index=False)
dic.to_csv(P / "master_cell_dictionary.csv", index=False)
for f in ["master_cell.csv", "master_cell_dictionary.csv"]:
    print(f"saved {f:28s} {pd.read_csv(P / f).shape}")

print("\none cell, a few columns:")
show = ["cell_id", "farm_pct", "winter_ndvi", "spring_gap_month", "ls_trend_class",
        "soil_limit_name", "si_wheat", "irrig_mm_wheat", "actual_share_pct_wheat"]
print(mc.loc[mc.cell_id == 300, show].T.to_string())

saved master_cell.csv              (560, 189)


saved master_cell_dictionary.csv   (189, 6)

one cell, a few columns:
                                             299
cell_id                                      300
farm_pct                                0.852128
winter_ndvi                              0.81434
spring_gap_month                               5
ls_trend_class                            stable
soil_limit_name         no or slight constraints
si_wheat                                   76.39
irrig_mm_wheat                             353.0
actual_share_pct_wheat                    19.558


## Conclusion

`master_cell.csv` is built: **560 rows x 189 columns**, one row per 1 km cell, with every source in it. Every check passed. Every number below comes from the outputs above.

### The merge was checked from a new angle

The 43 fixed columns we took from `master_monthly.csv` (elevation, slope, soil, GAEZ suitability, class and yield) are repeated 108 times per cell in that table. **Not one of them changes between months in any cell.** That is independent proof that every join in notebook 07 matched each cell to its own values.

Three more checks tie the new table back to earlier notebooks: the latitude and longitude match the grid exactly, the mean of the December-March columns reproduces `winter_ndvi` from notebook 08, and the lowest of the March-June columns reproduces the spring-gap NDVI. The pieces agree wherever they overlap.

### What is in it

| Source | Columns |
|---|---|
| GAEZ v5 (model) | 79 |
| Sentinel-2 | 39 |
| GAEZ v5 (actual 2020 harvest) | 16 |
| GPM | 13 |
| SMAP | 12 |
| NASA POWER | 12 |
| Landsat | 8 |
| SRTM | 5 |
| Grid | 5 |

**All 189 columns are documented** in `master_cell_dictionary.csv`, with unit, meaning, source dataset and the notebook that produced it. The dictionary is generated from the column names, and the notebook stops if any column is left undocumented, so the two files cannot drift apart.

### Missing values

Only **12 of 189** columns have any missing value, and both cases were found and explained earlier:

| Columns | Cells missing | Explained in |
|---|---|---|
| `si_*` for six crops | 40 | notebook 06: FAO has no suitability value there |
| `irrig_mm_*` for six crops | 22 | notebook 09: one GAEZ pixel outside its cropland map |

Nothing new is missing.

### The two integrated tables

| Table | Shape | One row = |
|---|---|---|
| `master_monthly.csv` | 60,480 x 76 | a cell in a month |
| `master_cell.csv` | 560 x 189 | a cell |

The scoring step reads **only** `master_cell.csv`.